**Cloning the repo so that we can use the private dataset for this test. The images are uploaded to the repo already!**

In [6]:
!git clone https://github.com/amanmandal7/COMP8410-A1_Evaluation_Presentation.git
%cd COMP8410-A1_Evaluation_Presentation/pair1_cv_recognition

Cloning into 'COMP8410-A1_Evaluation_Presentation'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 28 (delta 3), reused 22 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 793.94 KiB | 11.34 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/COMP8410-A1_Evaluation_Presentation/pair1_cv_recognition


In [7]:
!pip install deepface -q

In [8]:
from deepface import DeepFace
import os
import time

In [9]:
real = "data/real"
spoof = "data/spoof"

# Quick sanity check to see if files are actually there
print("real images:", sorted(os.listdir(real)))
print("spoof images:", sorted(os.listdir(spoof)))

real images: ['real_1.jpeg', 'real_2.jpeg', 'real_3.jpeg', 'real_4.jpeg']
spoof images: ['spoof_1.jpeg', 'spoof_2.jpeg', 'spoof_3.jpg', 'spoof_4.jpg']


In [10]:
# Running anti spoofing on every image

def check_spoof(folder):
    results = {}

    # sorting files
    files = sorted(os.listdir(folder))
    for fname in files:
        path = os.path.join(folder, fname)
        try:
            faces = DeepFace.extract_faces(
                img_path=path,
                anti_spoofing=True,
                enforce_detection=False
            )
            is_real_flag = faces[0]["is_real"]
            results[fname] = is_real_flag

        except Exception as e:
            results[fname] = None
            print("Uh oh, something broke on file:", fname, "->", e)

    return results

print("Starting anti-spoofing evaluation...")
start = time.time()

real_results = check_spoof(real)
spoof_results = check_spoof(spoof)

elapsed = time.time() - start

print("Real images results ->", real_results)
print("Spoof images results ->", spoof_results)

total_images = len(real_results) + len(spoof_results)
print("total time:", round(elapsed, 1), "s for", total_images, "images")
print("average time per image:", round(elapsed / total_images, 3), "s")

Starting anti-spoofing evaluation...
26-10-24 09:40:52 - 🔗 2.7_80x80_MiniFASNetV2.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/2.7_80x80_MiniFASNetV2.h5 to /root/.deepface/weights/2.7_80x80_MiniFASNetV2.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/2.7_80x80_MiniFASNetV2.h5
To: /root/.deepface/weights/2.7_80x80_MiniFASNetV2.h5
100%|██████████| 2.23M/2.23M [00:00<00:00, 37.3MB/s]


26-10-24 09:40:52 - 🔗 4_0_0_80x80_MiniFASNetV1SE.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/4_0_0_80x80_MiniFASNetV1SE.h5 to /root/.deepface/weights/4_0_0_80x80_MiniFASNetV1SE.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/4_0_0_80x80_MiniFASNetV1SE.h5
To: /root/.deepface/weights/4_0_0_80x80_MiniFASNetV1SE.h5
100%|██████████| 2.29M/2.29M [00:00<00:00, 40.7MB/s]


Real images results -> {'real_1.jpeg': True, 'real_2.jpeg': True, 'real_3.jpeg': False, 'real_4.jpeg': False}
Spoof images results -> {'spoof_1.jpeg': False, 'spoof_2.jpeg': False, 'spoof_3.jpg': False, 'spoof_4.jpg': False}
total time: 16.0 s for 8 images
average time per image: 1.994 s


**Now we compute the erorr rates and the Accuracy**

In [11]:
spoof_vals = []
for v in spoof_results.values():
    if v is not None:
        spoof_vals.append(v)

real_vals = []
for v in real_results.values():
    if v is not None:
        real_vals.append(v)

# APCER = Attack Presentation Classification Error Rate (spoofs wrongly accepted as real)
false_accepts = 0
for v in spoof_vals:
    if v == True:
        false_accepts += 1
apcer = false_accepts / len(spoof_vals)

# BPCER = Bona Fide Presentation Classification Error Rate (real faces wrongly rejected as spoof)
false_rejects = 0
for v in real_vals:
    if v == False:
        false_rejects += 1
bpcer = false_rejects / len(real_vals)

# ACER is just the average of the two error rates
acer = (apcer + bpcer) / 2

correct_real = sum(1 for v in real_vals if v == True)
correct_spoof = sum(1 for v in spoof_vals if v == False)
accuracy = (correct_real + correct_spoof) / (len(real_vals) + len(spoof_vals))

print(f"APCER (spoofs missed): {apcer:.2f}")
print(f"BPCER (real rejected): {bpcer:.2f}")
print(f"ACER (average error):  {acer:.2f}")
print(f"Accuracy:              {accuracy:.2f}")

APCER (spoofs missed): 0.00
BPCER (real rejected): 0.50
ACER (average error):  0.25
Accuracy:              0.75
